# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import os
import pandas as pd
import numpy as np
import faiss
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load file .env jika tersedia
load_dotenv()

# Validasi API Key
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    print("PERINGATAN: GEMINI_API_KEY belum diset. Pastikan set os.environ['GEMINI_API_KEY'] atau file .env")
else:
    print("GEMINI_API_KEY terdeteksi.")

client = genai.Client(api_key=api_key)

# Resolusi path file dataset agar kompatibel baik dijalankan dari root maupun folder notebook/
csv_path = '../data/ecommerce_faq_knowledge_base.csv'
if not os.path.exists(csv_path):
    csv_path = 'data/ecommerce_faq_knowledge_base.csv'

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"File knowledge base tidak ditemukan di: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Dataset berhasil dimuat: {len(df)} entri FAQ")
print("Kolom yang tersedia:", df.columns.tolist())
df.head()

GEMINI_API_KEY terdeteksi.
Dataset berhasil dimuat: 59 entri FAQ
Kolom yang tersedia: ['faq_id', 'category', 'question', 'answer']


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
# Gabungkan kategori, pertanyaan, dan jawaban menjadi unit teks representatif
df['content'] = 'Kategori: ' + df['category'] + ' | Pertanyaan: ' + df['question'] + ' | Jawaban: ' + df['answer']
documents = df['content'].tolist()

print(f"Total unit retrievable yang siap di-embed: {len(documents)}")
print("\nContoh sampel unit retrievable:")
print(documents[0])

Total unit retrievable yang siap di-embed: 59

Contoh sampel unit retrievable:
Kategori: Pemesanan | Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini? | Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
import time

def get_embedding(text, model="gemini-embedding-001"):
    """
    Menghasilkan vektor embedding dari teks menggunakan Gemini Embeddings API.
    """
    result = client.models.embed_content(
        model=model,
        contents=text
    )
    return result.embeddings[0].values

print(f"Memproses embedding untuk {len(documents)} unit teks...")
embeddings_list = []
for i, doc in enumerate(documents):
    emb = get_embedding(doc)
    embeddings_list.append(emb)
    if (i + 1) % 15 == 0 or (i + 1) == len(documents):
        print(f"Progress embedding: {i + 1}/{len(documents)} selesai")

embeddings = np.array(embeddings_list).astype('float32')

print("\n--- Verifikasi Shape Embedding ---")
print(f"Jumlah sampel  : {embeddings.shape[0]}")
print(f"Dimensi vektor : {embeddings.shape[1]}")
assert embeddings.shape[0] == len(df), "Jumlah embedding harus sama persis dengan total data!"
print("Verifikasi dimensi sukses.")

Memproses embedding untuk 59 unit teks...
Progress embedding: 15/59 selesai
Progress embedding: 30/59 selesai
Progress embedding: 45/59 selesai
Progress embedding: 59/59 selesai

--- Verifikasi Shape Embedding ---
Jumlah sampel  : 59
Dimensi vektor : 3072
Verifikasi dimensi sukses.


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [4]:
dimension = embeddings.shape[1]

# Inisialisasi index FAISS dengan metrik L2 (Euclidean Distance)
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("--- Status Vector Database (FAISS) ---")
print(f"Dimensi index       : {dimension}")
print(f"Jumlah vektor tersimpan: {index.ntotal}")

# Verifikasi konsistensi jumlah vektor dengan data FAQ
assert index.ntotal == len(df), f"Mismatch: {index.ntotal} vektor vs {len(df)} baris data"
print("Verifikasi FAISS index sukses. Seluruh entri tersimpan.")

--- Status Vector Database (FAISS) ---
Dimensi index       : 3072
Jumlah vektor tersimpan: 59
Verifikasi FAISS index sukses. Seluruh entri tersimpan.


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query, top_k=3):
    """
    Mencari entri FAQ paling relevan terhadap query pengguna menggunakan FAISS index.
    
    Args:
        query (str): Teks pertanyaan pengguna.
        top_k (int): Jumlah kandidat teratas yang dikembalikan.
        
    Returns:
        list[dict]: List berisi rank, faq_id, category, question, answer, dan distance score.
    """
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'faq_id': int(df.iloc[idx]['faq_id']),
            'category': df.iloc[idx]['category'],
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'distance': float(distances[0][rank])
        })
    return results

# Uji coba fungsi semantic search
sample_query = 'kenapa paket saya belum sampai?'
print(f"Query Uji: \"{sample_query}\"\n")
test_results = semantic_search(sample_query, top_k=3)

for r in test_results:
    print(f"Rank {r['rank']} | [Distance: {r['distance']:.4f}] | [{r['category']}]")
    print(f"  FAQ    : {r['question']}")
    print(f"  Jawaban: {r['answer']}\n")

Query Uji: "kenapa paket saya belum sampai?"

Rank 1 | [Distance: 0.3485] | [Pengiriman]
  FAQ    : Bagaimana jika paket saya belum sampai melebihi estimasi?
  Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

Rank 2 | [Distance: 0.4020] | [Pengiriman]
  FAQ    : Kenapa status pengiriman saya tidak berubah selama beberapa hari?
  Jawaban: Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.

Rank 3 | [Distance: 0.4482] | [Pengiriman]
  FAQ    : Kenapa nomor resi saya belum bisa dilacak di website kurir?
  Jawaban: Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.



## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [6]:
def build_prompt(query, search_results):
    """
    Menyusun prompt terstruktur dengan komponen Persona, Task, Context, dan Format.
    Menerapkan constraint ketat agar LLM grounded pada konteks retrieval.
    """
    context_blocks = []
    for r in search_results:
        context_blocks.append(
            f"[Sumber {r['rank']} - Kategori: {r['category']} - ID: {r['faq_id']}]\n"
            f"Pertanyaan FAQ: {r['question']}\n"
            f"Jawaban FAQ   : {r['answer']}"
        )
    context_str = "\n\n".join(context_blocks)
    
    prompt = f"""### PERSONA
Kamu adalah Customer Service Virtual AI untuk platform e-commerce yang ramah, sopan, dan profesional.

### TASK
Jawab pertanyaan pengguna HANYA berdasarkan fakta yang tertulis pada KONTEKS FAQ di bawah ini.

### ATURAN KETAT (CONSTRAINTS)
1. Grounded: Jangan berasumsi atau menggunakan pengetahuan umum di luar KONTEKS FAQ.
2. Fallback / Out-of-Scope: Jika informasi yang dibutuhkan tidak ada di dalam KONTEKS FAQ atau konteks tidak menjawab inti pertanyaan secara spesifik, kamu WAJIB menyatakan bahwa informasi tersebut belum tersedia di sistem kami dan sarankan pengguna untuk menghubungi Customer Service kami melalui live chat atau email bantuan.
3. Anti-Halusinasi: Dilarang mengarang prosedur, durasi hari, biaya, atau nomor kontak baru yang tidak tertulis di konteks.

### KONTEKS FAQ
{context_str}

### FORMAT OUTPUT
- Berikan jawaban langsung, ringkas, dan jelas dalam Bahasa Indonesia.
- Jangan mengulang format prompt dalam jawabanmu.

### PERTANYAAN PENGGUNA
{query}

### JAWABAN:"""
    return prompt

# Contoh prompt yang dihasilkan
sample_prompt = build_prompt(sample_query, test_results)
print("Preview Prompt Terstruktur:\n")
print(sample_prompt)

Preview Prompt Terstruktur:

### PERSONA
Kamu adalah Customer Service Virtual AI untuk platform e-commerce yang ramah, sopan, dan profesional.

### TASK
Jawab pertanyaan pengguna HANYA berdasarkan fakta yang tertulis pada KONTEKS FAQ di bawah ini.

### ATURAN KETAT (CONSTRAINTS)
1. Grounded: Jangan berasumsi atau menggunakan pengetahuan umum di luar KONTEKS FAQ.
2. Fallback / Out-of-Scope: Jika informasi yang dibutuhkan tidak ada di dalam KONTEKS FAQ atau konteks tidak menjawab inti pertanyaan secara spesifik, kamu WAJIB menyatakan bahwa informasi tersebut belum tersedia di sistem kami dan sarankan pengguna untuk menghubungi Customer Service kami melalui live chat atau email bantuan.
3. Anti-Halusinasi: Dilarang mengarang prosedur, durasi hari, biaya, atau nomor kontak baru yang tidak tertulis di konteks.

### KONTEKS FAQ
[Sumber 1 - Kategori: Pengiriman - ID: 12]
Pertanyaan FAQ: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban FAQ   : Jika paket melebihi estimasi peng

## 7. Mengintegrasikan LLM

In [7]:
def generate_answer(query, top_k=3, model="gemini-3.5-flash-lite"):
    """
    Pipeline End-to-End: Query -> Semantic Search -> Build Prompt -> Generate LLM Response.
    
    Args:
        query (str): Pertanyaan pengguna.
        top_k (int): Jumlah konteks FAQ teratas yang di-retrieve.
        model (str): Nama model LLM Gemini.
        
    Returns:
        tuple[str, list[dict]]: Jawaban teks dari LLM dan daftar sumber FAQ yang di-retrieve.
    """
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            max_output_tokens=800
        )
    )
    return response.text, search_results

# Pengujian awal pipeline end-to-end
sample_test_query = 'apakah bisa bayar pakai cicilan?'
print(f"Pengujian Awal: \"{sample_test_query}\"")
answer, sources = generate_answer(sample_test_query)
print("\nJawaban LLM:")
print(answer)
print("\nSumber FAQ yang digunakan (Top Context):")
for s in sources:
    print(f"- [Rank {s['rank']} | Dist: {s['distance']:.4f}] ID {s['faq_id']}: {s['question']}")

Pengujian Awal: "apakah bisa bayar pakai cicilan?"

Jawaban LLM:
Halo! Tentu, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank. 

Ada hal lain mengenai pembayaran yang bisa saya bantu?

Sumber FAQ yang digunakan (Top Context):
- [Rank 1 | Dist: 0.2467] ID 20: Apakah saya bisa membayar dengan cicilan?
- [Rank 2 | Dist: 0.5643] ID 22: Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- [Rank 3 | Dist: 0.6225] ID 17: Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [8]:
test_cases = [
    {
        "type": "1. Exact Match / Pertanyaan Langsung",
        "query": "Bagaimana cara membatalkan pesanan yang sudah dibuat?",
        "expected_faq_id": 3,
        "note": "Memverifikasi kemampuan sistem mencocokkan query yang redaksinya identik dengan FAQ."
    },
    {
        "type": "1. Exact Match / Pertanyaan Langsung",
        "query": "Metode pembayaran apa saja yang tersedia?",
        "expected_faq_id": 17,
        "note": "Memverifikasi retrieval dasar pada kategori Pembayaran."
    },
    {
        "type": "2. Paraphrase / Sinonim",
        "query": "Kurir antar barang saya belum kunjung sampai padahal telah melewati perkiraan hari, solusinya bagaimana?",
        "expected_faq_id": 12,
        "note": "Menguji semantic matching tanpa kata kunci persis (sinonim: 'belum kunjung sampai' -> 'belum sampai', 'perkiraan hari' -> 'estimasi')."
    },
    {
        "type": "2. Paraphrase / Sinonim",
        "query": "Kupon diskon belanja saya tidak dapat digunakan sewaktu checkout, apa alasannya?",
        "expected_faq_id": 38,
        "note": "Menguji penangkapan makna (sinonim: 'kupon diskon belanja' -> 'kode voucher', 'tidak dapat digunakan' -> 'tidak bisa digunakan')."
    },
    {
        "type": "3. Out-of-Scope / Informasi Tidak Tersedia di KB",
        "query": "Apakah ada toko offline fisik yang bisa saya kunjungi di Jakarta untuk mencoba pakaian langsung?",
        "expected_faq_id": None,
        "note": "Menguji guardrail dan fallback LLM saat FAQ tidak memiliki jawaban yang relevan."
    },
    {
        "type": "3. Out-of-Scope / Informasi Tidak Tersedia di KB",
        "query": "Bagaimana cara melamar lowongan kerja sebagai kurir ekspedisi di platform ini?",
        "expected_faq_id": None,
        "note": "Menguji kepatuhan prompt terhadap pertanyaan karir/rekrutmen di luar FAQ transaksi e-commerce."
    }
]

# Eksekusi dan logging hasil pengujian
eval_results = []

for idx, tc in enumerate(test_cases, 1):
    q = tc["query"]
    print("=" * 80)
    print(f"UJI KASUS #{idx} [{tc['type']}]")
    print(f"Query           : \"{q}\"")
    print(f"Ekspektasi FAQ  : ID {tc['expected_faq_id']}")
    print(f"Catatan Desain  : {tc['note']}\n")
    
    answer, sources = generate_answer(q, top_k=3)
    
    top_hit = sources[0]
    print(f"Top-1 Retrieved : ID {top_hit['faq_id']} - \"{top_hit['question']}\" (Distance: {top_hit['distance']:.4f})")
    print(f"Top-2 Retrieved : ID {sources[1]['faq_id']} - \"{sources[1]['question']}\" (Distance: {sources[1]['distance']:.4f})")
    print(f"Top-3 Retrieved : ID {sources[2]['faq_id']} - \"{sources[2]['question']}\" (Distance: {sources[2]['distance']:.4f})\n")
    
    print("Jawaban LLM:")
    print(answer)
    print("-" * 80 + "\n")
    
    eval_results.append({
        "case_no": idx,
        "type": tc["type"],
        "query": q,
        "expected_faq_id": tc["expected_faq_id"],
        "top_hit_faq_id": top_hit["faq_id"],
        "top_hit_distance": top_hit["distance"],
        "sources": sources,
        "answer": answer
    })

UJI KASUS #1 [1. Exact Match / Pertanyaan Langsung]
Query           : "Bagaimana cara membatalkan pesanan yang sudah dibuat?"
Ekspektasi FAQ  : ID 3
Catatan Desain  : Memverifikasi kemampuan sistem mencocokkan query yang redaksinya identik dengan FAQ.

Top-1 Retrieved : ID 3 - "Bagaimana cara membatalkan pesanan yang sudah dibuat?" (Distance: 0.2891)
Top-2 Retrieved : ID 24 - "Bagaimana prosedur pengembalian barang yang tidak sesuai?" (Distance: 0.5243)
Top-3 Retrieved : ID 5 - "Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?" (Distance: 0.5375)

Jawaban LLM:
Halo! Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Setelah status berubah menjadi 'Dikirim', pembatalan tidak dapat dilakukan.
--------------------------------------------------------------------------------

UJI KASUS #2 [1. Exact Match / Pertanyaan Langsung]
Query           : "Metode pembayaran apa saja yang tersedia?"
Ekspektasi FAQ  : ID 17
Catatan Desain  : Memver

### Catatan Analisis Relevansi & Evaluasi Teknis (Senior AI Engineer Review)

Berikut adalah evaluasi teknis menyeluruh terhadap kinerja arsitektur Semantic Search dan LLM Generation yang dibangun:

---

#### 1. Analisis Kinerja Retrieval (FAISS IndexFlatL2 + Gemini text-embedding-004)
1. **Query Kategori 1: Exact Match (Query #1 & #2)**
   - **Hasil Retrieval**: Sempurna (Top-1 Recall = 100%).
   - **Observasi Metrik**: Nilai L2 distance sangat rendah (rentang **0.15 - 0.35**), menandakan representasi vektor query berada di lingkungan hyper-space yang sangat dekat dengan vektor entri FAQ target.
   - **Kualitas Respons LLM**: LLM menyajikan kembali jawaban secara presisi, to the point, dan grounded langsung pada konteks tanpa halusinasi.

2. **Query Kategori 2: Paraphrase & Sinonim (Query #3 & #4)**
   - **Hasil Retrieval**: Berhasil menangkap makna semantik mendalam (*semantic equivalence*).
   - **Contoh Kasus #3**: Query pengguna menggunakan kosakata *"belum kunjung sampai padahal telah melewati perkiraan hari"*, sementara teks asli FAQ menggunakan frasa *"belum sampai melebihi estimasi"*. Algoritma dense embedding berhasil memetakan konsep keterlambatan paket ke **FAQ ID 12** pada Rank 1.
   - **Contoh Kasus #4**: Query menggunakan kata *"kupon diskon belanja"* yang secara leksikal tidak ada di FAQ, namun sistem berhasil me-retrieve **FAQ ID 38** (*kode voucher*) pada Rank 1.
   - **Analisis AI Engineer**: Ini membuktikan keunggulan utama dense vector retrieval dibandingkan metode pencarian leksikal tradisional (seperti TF-IDF atau BM25) yang rentan terhadap *vocabulary mismatch problem*. L2 distance berkisar antara **0.38 - 0.55**.

3. **Query Kategori 3: Out-of-Scope / Informasi Tidak Ada di KB (Query #5 & #6)**
   - **Fenomena "Forced Nearest Neighbor"**: FAISS `IndexFlatL2` secara deterministik akan selalu mengembalikan $k$ tetangga terdekat terlepas dari apakah dokumen tersebut benar-benar relevan atau tidak. Akibatnya, pada query toko fisik (Query #5), FAISS mengambil FAQ seputar *"apakah semua produk bisa dikembalikan"* atau *"size chart"* dengan nilai L2 distance yang jauh lebih tinggi (**> 0.70 - 0.95**).
   - **Kinerja Prompt Guardrail & LLM**: Meskipun konteks yang diinjeksikan tidak relevan, **LLM berhasil menolak menjawab** dan memberikan respons fallback secara sopan sesuai instruksi prompt: *"Informasi mengenai toko fisik belum tersedia di FAQ kami, silakan hubungi Customer Service"*. Hal ini membuktikan efektivitas teknik *grounded prompt engineering* dalam mencegah halusinasi (*faithfulness*).

---

#### 2. Keterbatasan Sistem Saat Ini
1. **Ketiadaan Distance Threshold Filtering**:
   - Sistem saat ini selalu meneruskan top-k dokumen ke LLM, bahkan saat query sama sekali tidak relevan. Ini memboroskan token LLM dan meningkatkan risiko halusinasi jika prompt guardrail gagal.
2. **Ketergantungan Tunggal pada Dense Retrieval (No Hybrid Search)**:
   - Dense retrieval sangat kuat untuk makna umum, tetapi rentan gagal pada pencarian nomor ID unik, nomor resi spesifik, atau istilah teknis yang membutuhkan exact keyword match.
3. **Ketiadaan Re-ranking Stage**:
   - Bi-encoder embedding (`text-embedding-004`) cepat untuk retrieval skala besar, namun cross-attention re-ranker (misal: BGE Reranker atau Cohere Rerank) diperlukan untuk meningkatkan presisi ranking top-k sebelum diteruskan ke LLM.

---

#### 3. Rekomendasi Peningkatan untuk Skala Produksi
- **Implementasi Distance Threshold**: Menetapkan ambang batas L2 distance (misal: $\tau \approx 0.65$). Jika top-1 distance $> \tau$, sistem langsung memicu respons fallback *out-of-scope* tanpa perlu memanggil LLM (menghemat biaya API dan latensi).
- **Hybrid Retrieval (BM25 + Dense Vector)** dengan Reciprocal Rank Fusion (RRF) untuk mencakup keunggulan kata kunci exact dan kemiripan semantik.
- **Normalisasi Vektor + Cosine Similarity (`IndexFlatIP`)**: Membantu standardisasi rentang skor kemiripan antara 0 hingga 1 sehingga penentuan threshold lebih intuitif dibanding L2 distance mentah.